<a href="https://colab.research.google.com/github/Reshsajee/my-project-resh/blob/main/mobilenetrealwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

In [ ]:
import os
import numpy as np
import cv2

X = []
y = []

dataset_path = "/content/drive/MyDrive/R11/MG"

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    images = sorted(os.listdir(folder_path))

    for i, img_name in enumerate(images):
        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))

        X.append(img)
        y.append(i)  # 0=pinhead,1=growing,2=mature

X = np.array(X)
y = np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train = tf.keras.applications.mobilenet_v2.preprocess_input(X_train)
X_test = tf.keras.applications.mobilenet_v2.preprocess_input(X_test)

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(3, activation='softmax')(x)

model_mobilenet = Model(inputs=base_model.input, outputs=output)

model_mobilenet.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_mobilenet.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model_mobilenet.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=8
)

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 400ms/step - accuracy: 0.5395 - loss: 1.0381 - val_accuracy: 0.6000 - val_loss: 0.8435
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 273ms/step - accuracy: 0.7895 - loss: 0.4070 - val_accuracy: 0.7000 - val_loss: 0.7917
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 296ms/step - accuracy: 0.8816 - loss: 0.3071 - val_accuracy: 0.7500 - val_loss: 0.5422
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step - accuracy: 0.9474 - loss: 0.1751 - val_accuracy: 0.6500 - val_loss: 1.2359
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 233ms/step - accuracy: 0.9079 - loss: 0.2496 - val_accuracy: 0.7000 - val_loss: 0.6771
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 241ms/step - accuracy: 0.9474 - loss: 0.1528 - val_accuracy: 0.6500 - val_loss: 1.1684
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 308ms/step - accuracy: 0.9605 - loss: 0.0991 - val_accuracy: 0.7000 - val_loss: 0.6610
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 246ms/step - accuracy: 0.9737 - loss: 0.0884 - val_accuracy: 0.

In [ ]:
loss, accuracy = model_mobilenet.evaluate(X_test, y_test)

print("✅ Test Accuracy:", accuracy * 100, "%")
print("✅ Test Loss:", loss)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 559ms/step - accuracy: 0.7500 - loss: 0.9700
✅ Test Accuracy: 75.0 %
✅ Test Loss: 0.9699695706367493


In [ ]:
for layer in model_mobilenet.layers[-30:]:
    layer.trainable = True

In [ ]:
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),  # low LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_fine = model_mobilenet.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8
)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 11s 424ms/step - accuracy: 0.5395 - loss: 2.0462 - val_accuracy: 0.7500 - val_loss: 0.9399
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 297ms/step - accuracy: 0.6316 - loss: 1.4785 - val_accuracy: 0.7000 - val_loss: 0.9177
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 369ms/step - accuracy: 0.6711 - loss: 1.0441 - val_accuracy: 0.7000 - val_loss: 0.9103
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 292ms/step - accuracy: 0.7237 - loss: 0.8797 - val_accuracy: 0.7000 - val_loss: 0.9069
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 294ms/step - accuracy: 0.7763 - loss: 0.7326 - val_accuracy: 0.7000 - val_loss: 0.9111
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 374ms/step - accuracy: 0.8289 - loss: 0.6376 - val_accuracy: 0.7000 - val_loss: 0.9158
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 288ms/step - accuracy: 0.7368 - loss: 0.6805 - val_accuracy: 0.7000 - val_loss: 0.9231
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 306ms/step - accuracy: 0.8816 - loss: 0.4375 - val_accuracy: 0

In [ ]:
import os
from datetime import datetime
import cv2
import numpy as np

X = []
y_time = []   # time to harvest (in hours)

dataset_path = "/content/drive/MyDrive/R11/MG"

for folder in sorted(os.listdir(dataset_path)):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    images = sorted(os.listdir(folder_path))

    times = []

    # Extract timestamps
    for img_name in images:
        time_str = img_name.split('_')[1] + "_" + img_name.split('_')[2].split('.')[0]
        time_obj = datetime.strptime(time_str, "%Y%m%d_%H%M%S")
        times.append(time_obj)

    harvest_time = times[-1]  # last image = mature

    # Loop again for training
    for i, img_name in enumerate(images):
        img_path = os.path.join(folder_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.resize(img, (224, 224))

        X.append(img)

        # ⏳ remaining time in HOURS
        remaining_time = (harvest_time - times[i]).total_seconds() / 3600.0
        y_time.append(remaining_time)

X = np.array(X)
y_time = np.array(y_time)

print("Loaded:", X.shape, y_time.shape)

Loaded: (96, 224, 224, 3) (96,)


In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import tensorflow as tf

# Preprocess
X = tf.keras.applications.mobilenet_v2.preprocess_input(X)

# Base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)

# 🔥 regression output
output = Dense(1, activation='linear')(x)

model_time = Model(inputs=base_model.input, outputs=output)

model_time.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model_time.summary()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_time, test_size=0.2, random_state=42
)

history = model_time.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=8
)

Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 290ms/step - loss: 446.9579 - mae: 15.9107 - val_loss: 1380.3778 - val_mae: 31.0017
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 238ms/step - loss: 412.6028 - mae: 15.1248 - val_loss: 1496.1658 - val_mae: 32.0057
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 249ms/step - loss: 358.3935 - mae: 13.6058 - val_loss: 1426.1283 - val_mae: 31.3624
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 242ms/step - loss: 327.5540 - mae: 13.3888 - val_loss: 1387.1998 - val_mae: 30.9895
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 358ms/step - loss: 320.4449 - mae: 13.1598 - val_loss: 1488.3909 - val_mae: 31.8983
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 258ms/step - loss: 268.3130 - mae: 12.0024 - val_loss: 1385.2252 - val_mae: 30.9775
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 238ms/step - loss: 265.2258 - mae: 12.1202 - val_loss: 1446.8933 - val_mae: 31.6510
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 250ms/step - loss: 224.7320 - mae: 11.0399 - val_loss: 1428.7041 - val_mae: 31.4402


In [ ]:
from google.colab import files
import cv2
import numpy as np
import tensorflow as tf

# 📤 Upload image
uploaded = files.upload()

# Get uploaded file name
img_path = list(uploaded.keys())[0]

# 🔍 Prediction function
def predict_time(img_path):
    # Read image
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))

    # Preprocess
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    # Predict time
    pred_time = model_time.predict(img)[0][0]

    # Convert to days + hours
    days = int(pred_time // 24)
    hours = int(pred_time % 24)

    print(f"⏳ Estimated Time to Harvest: {days} days {hours} hours")

# 🚀 Run prediction
predict_time(img_path)

In [ ]:
from google.colab import files
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# 📤 Upload image
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# 🔍 Prediction function
def predict_stage_and_time(img_path):
    # Read image
    img = cv2.imread(img_path)
    img_resized = cv2.resize(img, (224, 224))

    # Preprocess
    img_input = tf.keras.applications.mobilenet_v2.preprocess_input(img_resized)
    img_input = np.expand_dims(img_input, axis=0)

    # -------------------------
    # 🌱 Stage Prediction
    # -------------------------
    stage_pred = model_mobilenet.predict(img_input)
    class_idx = np.argmax(stage_pred)

    class_names = ['Pinhead', 'Growing', 'Mature']
    stage = class_names[class_idx]

    # -------------------------
    # ⏳ Time Prediction
    # -------------------------
    time_pred = model_time.predict(img_input)[0][0]

    days = int(time_pred // 24)
    hours = int(time_pred % 24)

    # -------------------------
    # 📊 Output
    # -------------------------
    print("🌱 Predicted Stage:", stage)

    if stage == "Mature":
        print("✅ Ready to Harvest!")
    else:
        print(f"⏳ Estimated Time: {days} days {hours} hours")

    # Show image
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Stage: {stage}")
    plt.axis('off')
    plt.show()


# 🚀 Run
predict_stage_and_time(img_path)